# Inferência estatística, correlação e regressão
Distribuição amostral da média e distribuição normal, intervalos de confiança, testes de hipótese (z, t, não paramétricos e qui-quadrado) e correlação de Pearson com regressão linear simples.

**Origem:** Alura, cursos de estatística com Python (amostragem, testes de hipótese, correlação e regressão).

**Dados:** envie para `/content/` no Colab: `dados_alturas.csv`, `dados_idade_aposentadoria.csv`, `dados_vida_lampada.csv` e `experimento_lampadas_natalinas.csv` (arquivos do curso), `dados.csv` (PNAD 2015) e `Desemprego.xlsx` (taxas de desemprego por UF). Os demais exemplos usam dados digitados no próprio notebook.

**Índice**
1. Amostragem e distribuições
2. Intervalos de confiança
3. Testes de hipótese
4. Correlação e regressão linear simples

In [ ]:
# Setup: todas as bibliotecas já vêm instaladas no Colab
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind, ttest_rel, wilcoxon

## 1. Amostragem e distribuições
Três populações com formas diferentes (alturas de funcionários, idade de aposentadoria e duração de lâmpadas) e o comportamento da média de muitas amostras retiradas delas.

### 1.1 Distribuições das populações

In [ ]:
dados_altura = pd.read_csv('/content/dados_alturas.csv')
dados_idade = pd.read_csv('/content/dados_idade_aposentadoria.csv')
dados_lampada = pd.read_csv('/content/dados_vida_lampada.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].hist(dados_altura['alturas'], bins=30, alpha=0.7, color='blue')
axes[0].set_title('Alturas')
axes[1].hist(dados_idade['idade'], bins=30, alpha=0.7, color='blue')
axes[1].set_title('Idade de aposentadoria')
axes[2].hist(dados_lampada['duracao'], bins=30, alpha=0.7, color='blue')
axes[2].set_title('Duração das lâmpadas')
plt.show()

### 1.2 Distribuição amostral da média
Retiramos `qtd` = 10.000 amostras de tamanho `n` = 100 (com reposição) de cada população e guardamos a média de cada amostra. Pelo **Teorema do Limite Central**, a distribuição dessas médias se aproxima de uma normal centrada na média populacional, qualquer que seja a forma da população.

In [ ]:
def calcula_media(dados, coluna, n, qtd):
    """Lista com as médias de `qtd` amostras de tamanho `n`."""
    return [dados[coluna].sample(n, replace=True).mean() for _ in range(qtd)]


n = 100
qtd = 10000

media_altura = calcula_media(dados_altura, 'alturas', n, qtd)
media_idade = calcula_media(dados_idade, 'idade', n, qtd)
media_lampada = calcula_media(dados_lampada, 'duracao', n, qtd)

In [ ]:
print('Médias populacionais')
print('  Idade:', dados_idade['idade'].mean())
print('  Altura:', dados_altura['alturas'].mean())
print('  Duração das lâmpadas:', dados_lampada['duracao'].mean())

print('Médias das médias amostrais')
print('  Idade:', np.mean(media_idade))
print('  Altura:', np.mean(media_altura))
print('  Duração das lâmpadas:', np.mean(media_lampada))

In [ ]:
plt.subplots(figsize=(15, 5))
plt.subplot(131)
plt.hist(media_idade, bins=30, alpha=0.7, color='blue')
plt.title('Idade média de aposentadoria')
plt.xlabel('Idade média de aposentadoria')
plt.ylabel('Frequência')

plt.subplot(132)
plt.hist(media_lampada, bins=30, alpha=0.7, color='purple')
plt.title('Duração média de uma lâmpada')
plt.xlabel('Tempo de vida médio (horas)')

plt.subplot(133)
plt.hist(media_altura, bins=30, alpha=0.7, color='green')
plt.title('Altura média dos funcionários')
plt.xlabel('Altura média (cm)')
plt.show()

### 1.3 Distribuição normal e erro padrão
Uma nova amostra de lâmpadas teve duração média de 1.200 horas. Quão distante essa média está do comportamento das médias amostrais?

O desvio padrão das médias amostrais é o **erro padrão** (EP). Na distribuição normal, cerca de 99,7% dos valores ficam a até 3 desvios da média.

In [ ]:
media_nova_amostra = 1200
media_duracao = dados_lampada['duracao'].mean()

duracao_amostras = pd.DataFrame({'media_duracao': media_lampada})
erro_padrao = duracao_amostras['media_duracao'].std()
print('Erro padrão:', erro_padrao)
print('Média das médias:', duracao_amostras['media_duracao'].mean())
print('Limite inferior (média - 3 EP):', media_duracao - 3 * erro_padrao)

In [ ]:
plt.hist(media_lampada, bins=30, alpha=0.7, color='purple')
plt.title('Duração média de uma lâmpada')
plt.xlabel('Tempo de vida médio (horas)')
plt.axvline(media_duracao, color='yellow')
plt.axvline(media_duracao + 3 * erro_padrao, color='green')
plt.axvline(media_duracao - 3 * erro_padrao, color='green')
plt.annotate('nova amostra', xy=(media_nova_amostra, 0), xytext=(media_nova_amostra, 100), fontsize=8,
             arrowprops=dict(facecolor='green'))
plt.show()

Quantas médias amostrais ficam entre média ± 3 EP, e quanto isso representa?

In [ ]:
dentro_3ep = duracao_amostras['media_duracao'].between(media_duracao - 3 * erro_padrao,
                                                        media_duracao + 3 * erro_padrao,
                                                        inclusive='neither')
print('Amostras dentro de ±3 EP:', dentro_3ep.sum())
print(f'Percentual: {dentro_3ep.mean() * 100:.2f}%')

## 2. Intervalos de confiança
Um intervalo de confiança dá a faixa de valores que, com um nível de confiança escolhido, deve conter o parâmetro populacional. Com a aproximação normal: $\bar{X} \pm z \cdot EP$, com $EP = \frac{\sigma}{\sqrt{n}}$.

### 2.1 Duração das lâmpadas (95%)

In [ ]:
desvio_padrao = dados_lampada['duracao'].std()
tamanho_amostra = len(dados_lampada)
erro_padrao = desvio_padrao / np.sqrt(tamanho_amostra)

confianca = 0.95
intervalo_confianca = stats.norm.interval(confianca, loc=media_duracao, scale=erro_padrao)
print('Erro padrão:', erro_padrao)
print('IC (95%):', intervalo_confianca)

### 2.2 Avaliações do TechTaste (90% e 95%)
Quanto maior o nível de confiança, mais largo o intervalo. O erro padrão vem de `stats.sem()` (desvio padrão amostral / $\sqrt{n}$).

In [ ]:
df_techtaste = pd.DataFrame({'avaliacoes': [38, 44, 33, 42, 47, 33, 36, 39, 42, 36, 39, 34, 42, 42, 36, 43, 31, 35, 36, 41, 42, 30, 25, 38, 47, 36, 32, 45, 44, 45, 37, 48, 37, 36, 44, 49, 31, 45, 45, 40, 36, 50, 38, 34, 36, 42, 46, 49, 36, 34, 38, 31, 53, 40, 57, 40, 36, 42, 26, 50, 32, 43, 35, 37, 42, 30, 36, 43, 40, 43, 44, 52, 37, 51, 35, 47, 40, 50, 37, 49]})
avaliacoes = df_techtaste['avaliacoes']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(avaliacoes, bins=10, alpha=0.7, color='blue')
axes[0].set_title('Distribuição das Avaliações')
axes[0].set_xlabel('Avaliação')
axes[0].set_ylabel('Frequência')
sns.kdeplot(avaliacoes, color='blue', fill=True, linewidth=2, ax=axes[1])
axes[1].set_title('Curva da Distribuição')
axes[1].set_xlabel('Avaliação')
plt.show()

In [ ]:
media_amostral = avaliacoes.mean()
erro_tech = stats.sem(avaliacoes)

print('Média amostral:', media_amostral)
print('Desvio padrão:', avaliacoes.std())
print('IC (90%):', stats.norm.interval(0.90, loc=media_amostral, scale=erro_tech))
print('IC (95%):', stats.norm.interval(0.95, loc=media_amostral, scale=erro_tech))

## 3. Testes de hipótese
Um teste confronta a **hipótese nula** ($H_0$, ausência de efeito ou diferença) com a **alternativa** ($H_1$). Rejeitamos $H_0$ quando o **p-valor** é menor que o **nível de significância** $\alpha$ (em geral 0,05 = 1 - nível de confiança).

- **Teste z:** amostras com mais de 30 observações e desvio padrão da população conhecido; amostra aleatória e observações independentes.
- **Teste t:** amostras pequenas (menos de 30) e desvio padrão populacional desconhecido; amostra aleatória e observações independentes.
  - **1 amostra:** hipótese sobre a média de uma amostra.
  - **2 amostras independentes:** hipótese sobre a diferença entre as médias de duas amostras.
  - **2 amostras pareadas (relacionadas):** a mesma amostra medida duas vezes.

In [ ]:
def decisao(p_valor, nivel_significancia=0.05):
    """Conclusão do teste a partir do p-valor."""
    if p_valor < nivel_significancia:
        return 'Rejeitar a hipótese nula'
    return 'Não rejeitar a hipótese nula'

### 3.1 Teste z
**TechTaste:** a média das avaliações é diferente de 30? ($H_0: \mu = 30$, bilateral). O intervalo em torno de $H_0$ é a região em que a média amostral não levaria à rejeição.

In [ ]:
h0 = 30
print(f'Região de não rejeição (95%) em torno de H0: {stats.norm.interval(0.95, loc=h0, scale=erro_tech)}')

z_score, p_value = sm.stats.ztest(avaliacoes, value=h0, alternative='two-sided')
print('Z-test:', z_score, 'p-value:', p_value)
print('Conclusão:', decisao(p_value))

**Lâmpadas natalinas:** o fabricante informa duração média de 1.570 horas, com desvio padrão populacional de 105 horas.

In [ ]:
lampada_natalina = pd.read_csv('/content/experimento_lampadas_natalinas.csv')

h0 = 1570
desvio = 105
tamanho = len(lampada_natalina['duracao'])
media_observada = lampada_natalina['duracao'].mean()

intervalo = stats.norm.interval(0.95, loc=h0, scale=desvio / np.sqrt(tamanho))
print('H0:', h0)
print('Média:', media_observada)
print('IC (95%) em torno de H0:', intervalo)

In [ ]:
z_stat, p_valor = sm.stats.ztest(lampada_natalina['duracao'], value=h0, alternative='two-sided')
print('Z-test:', z_stat, 'p-value:', p_valor)
print('Conclusão:', decisao(p_valor))

A estatística z calculada pela fórmula, com o desvio padrão populacional conhecido:

$$z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}}$$

In [ ]:
erro_padrao = desvio / np.sqrt(tamanho)
z = (media_observada - h0) / erro_padrao
print('Valor de Z:', z)

### 3.2 Teste t para uma amostra
**Suporte técnico:** o tempo médio de resposta de 25 solicitações é menor que 30 minutos? ($H_1: \mu < 30$, unilateral à esquerda).

In [ ]:
tempo_resposta = [28, 32, 29, 31, 30, 33, 28, 30, 31, 29,
                  30, 32, 29, 31, 30, 33, 28, 30, 31, 29,
                  30, 32, 24, 29, 30]

print('Média:', np.mean(tempo_resposta))
print('Desvio padrão:', np.std(tempo_resposta))
print('Erro padrão:', stats.sem(tempo_resposta))

t_stat, p_valor = stats.ttest_1samp(tempo_resposta, 30, alternative='less')
print('T-test:', t_stat, 'p-value:', p_valor)
print('Conclusão:', decisao(p_valor))

**Techsafe:** a porcentagem média de compressão é maior que 20%? ($H_1: \mu > 20$, unilateral à direita).

In [ ]:
df_techsafe = pd.DataFrame({'porcentagem_compressao': [21.99342831, 20.7234714, 22.29537708, 24.04605971, 20.53169325, 20.53172609, 24.15842563, 22.53486946, 20.06105123, 22.08512009, 20.07316461, 20.06854049, 21.48392454, 17.17343951, 17.55016433, 19.87542494, 18.97433776, 21.62849467, 19.18395185, 18.1753926, 23.93129754, 20.5484474, 21.13505641, 18.15050363, 19.91123455]})
compressao = df_techsafe['porcentagem_compressao']

h0 = 20
nivel_confianca = 0.95
nivel_significancia = 1 - nivel_confianca
erro = stats.sem(compressao)

t_test, p_value = stats.ttest_1samp(compressao, h0, alternative='greater')

print('Média:', compressao.mean())
print('Desvio padrão:', np.std(compressao))
print('IC (95%) em torno de H0:', stats.norm.interval(nivel_confianca, loc=h0, scale=erro))
print('T-test:', t_test, 'p-value:', p_value)
print('Decisão:', decisao(p_value, nivel_significancia))

### 3.3 Teste t para duas amostras independentes
**Propaganda:** as vendas com propaganda diferem das vendas sem propaganda?

In [ ]:
vendas_com_propaganda = [329.80, 291.70, 338.86, 391.38, 285.95, 285.95, 394.75, 346.05, 271.83, 332.55, 272.19, 272.06, 314.52, 185.20, 196.50, 266.26, 239.23, 318.85, 245.52, 215.26, 387.94, 286.45, 304.05, 214.52, 267.34, 306.66, 230.94, 322.54, 263.96, 282.50, 263.90, 411.14, 299.19, 236.54, 349.35, 226.75, 312.53, 182.42, 220.31, 311.81, 344.31, 310.28, 293.66, 281.93, 211.29, 256.81, 272.36, 363.43, 320.62, 194.22]
vendas_sem_propaganda = [304.44, 261.89, 244.38, 321.70, 346.86, 348.08, 234.65, 266.45, 304.88, 343.53, 256.25, 273.86, 218.62, 213.23, 333.75, 366.37, 280.68, 345.21, 306.70, 246.29, 306.68, 377.28, 282.85, 378.88, 127.82, 334.31, 290.22, 267.06, 290.51, 165.74, 271.82, 306.43, 373.67, 253.90, 236.49, 254.89, 339.92, 304.73, 253.21, 315.80, 292.80, 343.12, 242.88, 265.34, 261.47, 197.19, 302.77, 300.66, 285.31, 270.92]

print('Média com propaganda:', np.mean(vendas_com_propaganda))
print('Média sem propaganda:', np.mean(vendas_sem_propaganda))

plt.hist(vendas_com_propaganda, color='blue', alpha=0.5, label='com propaganda')
plt.hist(vendas_sem_propaganda, color='yellow', alpha=0.5, label='sem propaganda')
plt.xlabel('Vendas')
plt.ylabel('Frequência')
plt.title('Distribuição de Vendas com e sem propaganda')
plt.legend()
plt.show()

In [ ]:
t_statistic, p_value = ttest_ind(vendas_com_propaganda, vendas_sem_propaganda)
print('T-statistic:', t_statistic)
print('P-value:', p_value)
print('Decisão:', decisao(p_value))

A estatística t pela fórmula (com as variâncias amostrais):

$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}}$$

Com amostras de mesmo tamanho, o valor coincide com o do `ttest_ind`.

In [ ]:
amostra_1 = np.array(vendas_com_propaganda)
amostra_2 = np.array(vendas_sem_propaganda)

t_manual = (amostra_1.mean() - amostra_2.mean()) / np.sqrt(
    amostra_1.var(ddof=1) / len(amostra_1) + amostra_2.var(ddof=1) / len(amostra_2)
)
t_manual

**Filiais:** as avaliações médias das filiais Centro-Norte e Sul são diferentes?

In [ ]:
df_filiais = pd.DataFrame({'Filial Centro-Norte': [3.2, 2.9, 2.0, 3.3, 3.1],
                           'Filial Sul': [3.8, 4.0, 4.7, 4.9, 4.8]})

stat, p_valor = ttest_ind(df_filiais['Filial Centro-Norte'], df_filiais['Filial Sul'])
print(f'Estatística do teste t independente: {stat}')
print(f'Valor-p: {p_valor}')
print('Conclusão:', decisao(p_valor))

### 3.4 Teste t para duas amostras pareadas
A mesma amostra medida duas vezes (coleta A e coleta B). **Treinamento:** as notas aumentaram depois do treinamento? ($H_1$: depois > antes).

In [ ]:
antes_do_treinamento = [78, 65, 75, 80, 85, 88, 79, 81, 76, 84]
apos_o_treinamento = [85, 70, 75, 87, 90, 92, 84, 82, 76, 91]

resultado_subtracao = [b - a for a, b in zip(antes_do_treinamento, apos_o_treinamento)]
print(f'Diferença entre depois e antes: {resultado_subtracao}')

t_statistic, p_value = ttest_rel(apos_o_treinamento, antes_do_treinamento, alternative='greater')
print('T-statistic:', t_statistic)
print('P-value:', p_value)
print('Conclusão:', decisao(p_value))

**Equipe de vendas:** as vendas de cada vendedor mudaram depois da ação?

In [ ]:
df_equipe_vendas = pd.DataFrame({'Vendedor': ['Luíza', 'Bia', 'Rodrigo', 'Allan', 'Evaldo'],
                                 'Vendas Antes (R$)': [252.72, 203.91, 307.32, 185.78, 220.5],
                                 'Vendas Depois (R$)': [285.1, 223.15, 324.41, 202.23, 240.63]})

stat, p_valor = ttest_rel(df_equipe_vendas['Vendas Depois (R$)'], df_equipe_vendas['Vendas Antes (R$)'])
print(f'Estatística do teste t pareado: {stat}')
print(f'Valor-p: {p_valor}')
print('Conclusão:', decisao(p_valor))

### 3.5 Testes não paramétricos
Usados quando os pressupostos do teste t (como a normalidade) não se sustentam; comparam postos/medianas.

**Mann-Whitney:** duas amostras independentes (a variável A não representa as mesmas unidades de coleta que a B).

In [ ]:
treino_a = [5, 3, 8, 4, 6]
treino_b = [4, 2, 7, 5, 3]

stat, p_valor = mannwhitneyu(treino_a, treino_b, alternative='two-sided')
print('p-valor:', p_valor)
print('Conclusão:', decisao(p_valor))

In [ ]:
# H0: não há diferença entre as avaliações | H1: há diferença
df_avaliacoes = pd.DataFrame({
    'playcatch': [4, 5, 3, 4, 5, 4, 4, 3, 5, 4, 5, 3, 4],
    'concorrente': [3, 4, 3, 3, 2, 4, 3, 2, 3, 3, 4, 2, 3]
})

stat, p_valor = mannwhitneyu(df_avaliacoes['playcatch'], df_avaliacoes['concorrente'], alternative='two-sided')
print(f'Estatística do teste de Mann-Whitney: {stat}')
print(f'Valor-p: {p_valor}')
print('Conclusão:', decisao(p_valor))

**Wilcoxon:** duas amostras pareadas, com uso da mediana. Preferências dos consumidores antes e depois da mudança no layout:

In [ ]:
preferencias_antes = [4, 2, 5, 2, 5, 6, 7, 8, 9, 10]
preferencias_depois = [8, 5, 3, 5, 8, 9, 9, 9, 10, 9]

resultado_subtracao = [b - a for a, b in zip(preferencias_antes, preferencias_depois)]
print('Média das diferenças:', np.mean(resultado_subtracao))
print('Mediana das diferenças:', np.median(resultado_subtracao))

stat, p_valor = wilcoxon(preferencias_depois, preferencias_antes, alternative='greater')
print('p-valor:', p_valor)
print('Conclusão:', decisao(p_valor))

In [ ]:
df_wt = pd.DataFrame({
    'Antes_WatchTogether': [3, 4, 3, 4, 2, 4, 3, 2, 3, 3, 4, 2, 3],
    'Depois_WatchTogether': [4, 5, 3, 4, 5, 4, 4, 3, 5, 4, 5, 3, 4]
})

stat, p_valor = wilcoxon(df_wt['Depois_WatchTogether'] - df_wt['Antes_WatchTogether'])
print(f'Estatística do teste de Wilcoxon: {stat}')
print(f'Valor-p: {p_valor}')
print('Conclusão:', decisao(p_valor))

### 3.6 Teste qui-quadrado
Associação entre variáveis categóricas, a partir da tabela de contingência. O desempenho dos alunos depende do método de ensino?

In [ ]:
df_ensino = pd.DataFrame({'Desempenho': {0: 'Regular', 1: 'Bom', 2: 'Bom', 3: 'Excelente', 4: 'Bom', 5: 'Bom', 6: 'Regular', 7: 'Regular', 8: 'Excelente', 9: 'Ruim', 10: 'Bom', 11: 'Regular', 12: 'Muito Bom', 13: 'Bom', 14: 'Bom', 15: 'Regular', 16: 'Muito Bom', 17: 'Bom', 18: 'Muito Bom', 19: 'Excelente', 20: 'Ruim', 21: 'Excelente', 22: 'Ruim', 23: 'Ruim', 24: 'Regular', 25: 'Excelente', 26: 'Ruim', 27: 'Excelente', 28: 'Excelente', 29: 'Regular', 30: 'Regular', 31: 'Ruim', 32: 'Regular', 33: 'Regular', 34: 'Muito Bom', 35: 'Excelente', 36: 'Regular', 37: 'Excelente', 38: 'Excelente', 39: 'Muito Bom', 40: 'Ruim', 41: 'Ruim', 42: 'Ruim', 43: 'Ruim', 44: 'Muito Bom', 45: 'Regular', 46: 'Excelente', 47: 'Muito Bom', 48: 'Muito Bom', 49: 'Muito Bom', 50: 'Muito Bom', 51: 'Muito Bom', 52: 'Bom', 53: 'Ruim', 54: 'Muito Bom', 55: 'Bom', 56: 'Bom', 57: 'Excelente', 58: 'Bom', 59: 'Muito Bom'},
                          'Metodo_Ensino': {0: 'Moderno', 1: 'Tradicional', 2: 'Tradicional', 3: 'Moderno', 4: 'Moderno', 5: 'Moderno', 6: 'Moderno', 7: 'Moderno', 8: 'Moderno', 9: 'Tradicional', 10: 'Tradicional', 11: 'Moderno', 12: 'Tradicional', 13: 'Moderno', 14: 'Tradicional', 15: 'Tradicional', 16: 'Moderno', 17: 'Moderno', 18: 'Tradicional', 19: 'Tradicional', 20: 'Moderno', 21: 'Tradicional', 22: 'Moderno', 23: 'Moderno', 24: 'Tradicional', 25: 'Tradicional', 26: 'Moderno', 27: 'Moderno', 28: 'Moderno', 29: 'Moderno', 30: 'Moderno', 31: 'Moderno', 32: 'Tradicional', 33: 'Moderno', 34: 'Tradicional', 35: 'Tradicional', 36: 'Tradicional', 37: 'Moderno', 38: 'Tradicional', 39: 'Moderno', 40: 'Tradicional', 41: 'Moderno', 42: 'Tradicional', 43: 'Moderno', 44: 'Moderno', 45: 'Moderno', 46: 'Moderno', 47: 'Moderno', 48: 'Moderno', 49: 'Tradicional', 50: 'Moderno', 51: 'Moderno', 52: 'Moderno', 53: 'Moderno', 54: 'Tradicional', 55: 'Moderno', 56: 'Moderno', 57: 'Tradicional', 58: 'Moderno', 59: 'Tradicional'}})

tab_contingencia = pd.crosstab(df_ensino['Desempenho'], df_ensino['Metodo_Ensino'])
chi2, p_valor, _, _ = chi2_contingency(tab_contingencia)

print(f'Estatística chi2: {chi2}')
print(f'p-valor: {p_valor}')
print('Conclusão:', decisao(p_valor))

## 4. Correlação e regressão linear simples
### 4.1 Covariância e coeficiente de correlação de Pearson
A **covariância** (variância conjunta) mede o grau de interdependência numérica entre duas variáveis.

**Covariância populacional:** $$\sigma_{xy} = \frac 1n\sum_{i=1}^{n}(X_i-\mu_x)(Y_i-\mu_y)$$

**Covariância amostral:** $$S_{xy} = \frac 1{n-1}\sum_{i=1}^{n}(X_i-\bar{X})(Y_i-\bar{Y})$$

$S_{xy}$ positivo indica associação linear positiva (x aumenta, y aumenta; pontos mais influentes nos quadrantes I e III). $S_{xy}$ negativo indica associação linear negativa (quadrantes II e IV). Pontos distribuídos uniformemente pelos quadrantes levam $S_{xy}$ para perto de zero: não há associação linear.

<img src="https://caelum-online-public.s3.amazonaws.com/1273-estatistica-parte4/01/img026.png" width=50%>

O **coeficiente de correlação** é a covariância dividida pelo produto dos desvios padrão de x e de y. Varia de -1 a +1: valores próximos de -1 ou +1 indicam relação linear forte; perto de zero, relação fraca. Correlação mede associação linear, **não causação**.

**Pearson, dados populacionais:** $$\rho_{xy} = \frac{\sigma_{xy}}{\sigma_{x}\sigma_{y}}$$

**Pearson, dados amostrais:** $$r_{xy} = \frac{s_{xy}}{s_{x}s_{y}}$$

onde $\sigma_{xy}$ e $s_{xy}$ são as covariâncias populacional e amostral, e $\sigma_{x}, \sigma_{y}$ e $s_{x}, s_{y}$ os desvios padrão populacionais e amostrais de x e y.

**Exemplo com a PNAD 2015:** renda e altura.

In [ ]:
dados = pd.read_csv('dados.csv')

x = dados.Renda
y = dados.Altura

ax = sns.scatterplot(x=x, y=y)
ax.set_title('Altura x Renda', fontsize=20)
ax.set_xlabel('Renda (R$)', fontsize=16)
ax.set_ylabel('Altura (m)', fontsize=16)
ax.figure.set_size_inches(10, 6)
ax.hlines(y=y.mean(), xmin=x.min(), xmax=x.max(), colors='black', linestyles='dashed')
ax.vlines(x=x.mean(), ymin=y.min(), ymax=y.max(), colors='black', linestyles='dashed')
plt.show()

In [ ]:
dados[['Altura', 'Renda']].corr()

Os pontos se espalham pelos quatro quadrantes: correlação praticamente nula. Em uma amostra pequena (10 pessoas com renda abaixo de R\$ 5.000), as correlações entre as variáveis podem parecer fortes por acaso:

In [ ]:
amostra = dados.query('Renda < 5000').sample(n=10, random_state=101)
amostra.corr()

**Gasto x Renda familiar** (R\$): dataset do curso com 50 observações.

In [ ]:
dataset = pd.DataFrame({
    'Y': [3011, 1305, 1879, 2654, 2849, 1068, 2892, 2543, 3074, 849, 2184, 2943, 1357, 2755, 2163, 3099, 1600, 353, 1778, 740, 2129, 3302, 2412, 2683, 2515, 2395, 2292, 1000, 600, 1864, 3027, 1978, 2791, 1982, 900, 1964, 1247, 3067, 700, 1500, 3110, 2644, 1378, 2601, 501, 1292, 2125, 1431, 2260, 1770],
    'X': [9714, 3728, 6062, 8845, 8378, 3338, 8507, 7947, 9915, 1632, 6825, 8918, 4100, 9184, 6180, 9997, 4500, 1069, 5925, 2466, 6083, 9712, 7780, 8383, 7185, 7483, 7640, 2100, 2000, 6012, 8902, 5345, 8210, 5662, 2700, 6546, 2900, 9894, 1500, 5000, 8885, 8813, 3446, 7881, 1164, 3401, 6641, 3329, 6648, 4800]
})

ax = sns.boxplot(dataset, orient='h', width=0.5)
ax.figure.set_size_inches(10, 6)
ax.set_title('Box plot', fontsize=20)
ax.set_xlabel('Reais (R$)', fontsize=16)
plt.show()

Pearson pela fórmula ($r_{xy} = s_{xy} / (s_x s_y)$), comparado com `corr()`:

In [ ]:
s_xy = dataset[['X', 'Y']].cov().loc['X', 'Y']
s_x = dataset['X'].std()
s_y = dataset['Y'].std()

print('Covariância amostral:', s_xy)
print('r pela fórmula:', s_xy / (s_x * s_y))
dataset.corr()

### 4.2 Regressão linear simples
Modelo $Y_i = \beta_1 + \beta_2 X_i + u_i$, estimado por mínimos quadrados ordinários (MQO).

**Obter $\hat{\beta}_2$ (coeficiente angular):**

$$\hat{\beta}_2 = \frac{n\sum{X_iY_i} - \sum{X_i}\sum{Y_i}}{n\sum{X_i^2} - (\sum{X_i})^2}$$

**Obter $\hat{\beta}_1$ (intercepto):**

$$\begin{eqnarray}\hat{\beta}_1 &=& \frac{\sum{X_i^2}\sum{Y_i} - \sum{X_i}\sum{X_iY_i}}{n\sum{X_i^2} - (\sum{X_i})^2}&=& \bar{Y} - \hat{\beta}_2\bar{X}\end{eqnarray}$$

In [ ]:
n = len(dataset)
SOMA_X = dataset['X'].sum()
SOMA_Y = dataset['Y'].sum()
SOMA_X2 = (dataset['X'] ** 2).sum()
SOMA_XY = (dataset['X'] * dataset['Y']).sum()

beta_2 = (n * SOMA_XY - SOMA_X * SOMA_Y) / (n * SOMA_X2 - SOMA_X ** 2)
beta_1 = dataset['Y'].mean() - beta_2 * dataset['X'].mean()
print('beta_1 (intercepto):', beta_1)
print('beta_2 (angular):', beta_2)

Os mesmos coeficientes com o `statsmodels` (OLS), com o resumo completo do modelo:

In [ ]:
Y = dataset['Y']
X = sm.add_constant(dataset['X'])
modelo = sm.OLS(Y, X).fit()
print(modelo.summary())

In [ ]:
dataset['Y_previsto'] = modelo.predict()
dataset.head()

In [ ]:
ax = sns.lmplot(data=dataset, x='X', y='Y')
ax.fig.set_size_inches(10, 6)
ax.fig.suptitle('Reta de Regressão - Gasto x Renda', fontsize=16, y=1.05)
ax.set_xlabels('Renda (R$)', fontsize=14)
ax.set_ylabels('Gasto (R$)', fontsize=14)
plt.show()

### 4.3 Matriz de correlação: desemprego por UF
Correlação entre as taxas de desemprego de vários estados, do Brasil e a rotatividade de Risotolândia, a partir de junho de 2022, em um heatmap interativo (Plotly) e nas séries no tempo.

In [ ]:
df_desemprego = pd.read_excel('Desemprego.xlsx')
df_filtered = df_desemprego[df_desemprego['Data'] >= '2022-06-30'].copy()
df_filtered.head()

In [ ]:
corr = df_filtered.drop('Data', axis=1).corr()

fig = go.Figure()
fig.add_trace(
    go.Heatmap(
        x=corr.columns,
        y=corr.index,
        z=np.array(corr),
        text=corr.values,
        texttemplate='%{text:.3f}',
        colorscale='inferno'))
fig.update_layout(
    height=750,
    width=750,
    yaxis=dict(autorange='reversed'))
fig.show()

In [ ]:
series = {
    'Paraná': 'brown', 'Santa Catarina': 'grey', 'São Paulo': 'purple', 'Rio de Janeiro': 'magenta',
    'Mato Grosso do Sul': 'blue', 'Minas Gerais': 'yellow', 'Brasil': 'black'
}

plt.figure(figsize=(12, 6))
for coluna, cor in series.items():
    plt.plot(df_filtered['Data'], df_filtered[coluna], label=coluna, color=cor, linestyle=':')
plt.plot(df_filtered['Data'], df_filtered['Risotolandia'], label='Risotolândia', color='orange', linewidth=3)
plt.title('Taxa de desemprego no Brasil x Rotatividade Risotolândia')
plt.xlabel('Data')
plt.ylabel('Taxas')
plt.legend()
plt.xticks(rotation=90)
plt.grid(True)
plt.show()